# Scenario Instantiation: Archetype → Scenarios with Controlled Uncertainty

**Duration:** 20 minutes  
**Level:** Intermediate  
**Prerequisites:** Complete [Notebook 00: Quick Start](00_quickstart.ipynb)

## Overview

This notebook demonstrates the **core methodological contribution** of BASICS-CDSS: transforming clinical archetypes into multiple test scenarios with **controlled uncertainty perturbations**.

### What You'll Learn:

1. How archetypes become scenarios
2. Five perturbation operators (Table 1 from manuscript)
3. Uncertainty profile quantification
4. Reproducible scenario generation with seeds
5. Stratified instantiation for ablation studies

### Why This Matters:

Real-world clinical data is messy: missing values, measurement noise, conflicting information. BASICS-CDSS systematically tests how CDSS systems behave under these conditions **before** deployment.

## 1. Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import asdict

# Import BASICS-CDSS scenario module
from basics_cdss.scenario import (
    instantiate_scenarios,
    instantiate_stratified_scenarios,
    create_default_perturbation,
    PerturbationConfig,
    Scenario
)
from basics_cdss.visualization.scenario_plots import (
    plot_uncertainty_distribution,
    plot_perturbation_effects
)

print("✓ BASICS-CDSS scenario module loaded")

## 2. Create Sample Archetypes

An **archetype** is a prototypical clinical case (from SynDX). We'll create three representative archetypes for demonstration.

In [ ]:
# Create sample archetypes (normally from SynDX CSV)
archetypes_data = [
    {
        'archetype_id': 'A001',
        'triage_tier': 'high',
        'age': 65,
        'heart_rate': 125,
        'temperature': 39.2,
        'blood_pressure_sys': 165,
        'blood_pressure_dia': 95,
        'respiratory_rate': 24,
        'oxygen_saturation': 92,
        'symptom_chest_pain': 1,
        'symptom_shortness_of_breath': 1,
    },
    {
        'archetype_id': 'A002',
        'triage_tier': 'medium',
        'age': 45,
        'heart_rate': 95,
        'temperature': 37.8,
        'blood_pressure_sys': 135,
        'blood_pressure_dia': 85,
        'respiratory_rate': 18,
        'oxygen_saturation': 96,
        'symptom_headache': 1,
        'symptom_fatigue': 1,
    },
    {
        'archetype_id': 'A003',
        'triage_tier': 'low',
        'age': 28,
        'heart_rate': 72,
        'temperature': 36.8,
        'blood_pressure_sys': 118,
        'blood_pressure_dia': 78,
        'respiratory_rate': 14,
        'oxygen_saturation': 98,
        'symptom_minor_cough': 1,
    },
]

archetypes_df = pd.DataFrame(archetypes_data)

print("Sample Archetypes:")
print(archetypes_df[['archetype_id', 'triage_tier', 'age', 'heart_rate', 'temperature']].to_string())
print(f"\nTotal archetypes: {len(archetypes_df)}")

## 3. Perturbation Operators (Table 1 from Manuscript)

BASICS-CDSS implements **five perturbation operators** to simulate real-world uncertainty:

| Operator | Type | Simulates |
|----------|------|-----------||
| **Mask** | Information Missingness | Missing lab results, incomplete records |
| **Noise** | Ambiguity | Measurement error, inter-observer variability |
| **Conflict** | Internal Inconsistency | Contradictory findings |
| **Degrade** | Reduced Specificity | Vague symptoms, imprecise descriptions |
| **Composite** | Combined | Multiple uncertainty types simultaneously |

Let's demonstrate each operator on a single archetype.

### 3.1 Mask Operator: Information Missingness

In [ ]:
# Configure mask operator with 30% missingness probability
mask_config = PerturbationConfig(p_mask=0.3)

# Apply mask operator to first archetype
mask_operator = create_default_perturbation("mask", config=mask_config, seed=42)
baseline_features = archetypes_df.iloc[0].drop(['archetype_id', 'triage_tier']).to_dict()

masked_features, uncertainty_profile = mask_operator.apply(baseline_features)

print("MASK OPERATOR: Information Missingness")
print("="*60)
print(f"Baseline features: {len(baseline_features)}")
print(f"Masked features: {len(masked_features)}")
print(f"Missingness rate: {uncertainty_profile['missingness']:.2%}")
print("\nRemoved features:")
removed = set(baseline_features.keys()) - set(masked_features.keys())
print(f"  {', '.join(removed)}" if removed else "  (none)")
print("\nRetained features:")
for key in list(masked_features.keys())[:5]:
    print(f"  {key}: {masked_features[key]}")

### 3.2 Noise Operator: Measurement Ambiguity

In [ ]:
# Configure noise operator with sigma=0.1
noise_config = PerturbationConfig(noise_sigma=0.1)

noise_operator = create_default_perturbation("noise", config=noise_config, seed=42)
noisy_features, uncertainty_profile = noise_operator.apply(baseline_features)

print("NOISE OPERATOR: Measurement Ambiguity")
print("="*60)
print(f"Ambiguity level: {uncertainty_profile['ambiguity']:.4f}")
print("\nFeature perturbations (showing numeric features):")
print(f"{'Feature':<25} {'Original':>12} {'Noisy':>12} {'Change':>12}")
print("-"*65)
for key in ['heart_rate', 'temperature', 'blood_pressure_sys', 'oxygen_saturation']:
    if key in baseline_features and key in noisy_features:
        orig = baseline_features[key]
        noisy = noisy_features[key]
        change = noisy - orig
        print(f"{key:<25} {orig:>12.2f} {noisy:>12.2f} {change:>+12.3f}")

### 3.3 Conflict Operator: Internal Inconsistency

The conflict operator requires defining contradiction mappings. For demonstration, we'll define conflicts between vital signs.

In [ ]:
# Configure conflict operator with contradiction rules
conflict_config = PerturbationConfig(
    conflict_pairs={
        'symptom_chest_pain': {1: 0, 0: 1},  # Flip chest pain indicator
        'symptom_shortness_of_breath': {1: 0, 0: 1},  # Flip shortness of breath
    }
)

conflict_operator = create_default_perturbation("conflict", config=conflict_config, seed=42)
conflicted_features, uncertainty_profile = conflict_operator.apply(baseline_features)

print("CONFLICT OPERATOR: Internal Inconsistency")
print("="*60)
print(f"Conflict rate: {uncertainty_profile['conflict']:.2%}")
print("\nIntroduced conflicts:")
for key in conflict_config.conflict_pairs.keys():
    if key in baseline_features:
        orig = baseline_features.get(key)
        conf = conflicted_features.get(key)
        if orig != conf:
            print(f"  {key}: {orig} → {conf} (CONFLICTED)")
        else:
            print(f"  {key}: {orig} (unchanged)")

### 3.4 Composite Perturbation: Combined Uncertainty

The composite operator applies multiple perturbations sequentially, simulating realistic multi-dimensional uncertainty.

In [ ]:
# Configure composite perturbation
composite_config = PerturbationConfig(
    p_mask=0.2,
    noise_sigma=0.08,
    conflict_pairs={'symptom_chest_pain': {1: 0, 0: 1}}
)

composite_operator = create_default_perturbation("composite", config=composite_config, seed=42)
composite_features, uncertainty_profile = composite_operator.apply(baseline_features)

print("COMPOSITE OPERATOR: Multi-dimensional Uncertainty")
print("="*60)
print("Uncertainty Profile:")
for metric, value in uncertainty_profile.items():
    print(f"  {metric.capitalize():<15}: {value:.4f}")

print(f"\nFeature count: {len(baseline_features)} → {len(composite_features)}")
print(f"Features removed: {len(baseline_features) - len(composite_features)}")

## 4. Scenario Instantiation with `instantiate_scenarios()`

Now let's use the main API function to generate multiple scenarios from our archetypes.

### 4.1 Baseline Scenarios (No Perturbation)

In [ ]:
# Generate baseline scenarios (no perturbation)
baseline_scenarios = instantiate_scenarios(
    archetypes_df,
    n_per_archetype=5,
    seed=42,
    apply_perturbation=False
)

print(f"Generated {len(baseline_scenarios)} baseline scenarios")
print(f"Scenarios per archetype: {len(baseline_scenarios) // len(archetypes_df)}")

# Examine first scenario
print("\nFirst baseline scenario:")
s = baseline_scenarios[0]
print(f"  Archetype ID: {s.archetype_id}")
print(f"  Seed: {s.seed}")
print(f"  Target tier: {s.targets['triage_tier']}")
print(f"  Feature count: {len(s.features)}")
print(f"  Uncertainty profile: {s.uncertainty_profile}")

### 4.2 Perturbed Scenarios with Composite Operator

In [ ]:
# Generate scenarios with composite perturbation
composite_scenarios = instantiate_scenarios(
    archetypes_df,
    n_per_archetype=10,
    seed=42,
    perturbation_type="composite",
    perturbation_config=PerturbationConfig(p_mask=0.25, noise_sigma=0.1)
)

print(f"Generated {len(composite_scenarios)} composite scenarios")
print(f"Scenarios per archetype: {len(composite_scenarios) // len(archetypes_df)}")

# Examine uncertainty profiles
print("\nUncertainty profile statistics:")
missingness = [s.uncertainty_profile['missingness'] for s in composite_scenarios]
ambiguity = [s.uncertainty_profile['ambiguity'] for s in composite_scenarios]
print(f"  Missingness: μ={np.mean(missingness):.3f}, σ={np.std(missingness):.3f}")
print(f"  Ambiguity:   μ={np.mean(ambiguity):.3f}, σ={np.std(ambiguity):.3f}")

# Show sample scenarios from different archetypes
print("\nSample scenarios by archetype:")
for arch_id in ['A001', 'A002', 'A003']:
    arch_scenarios = [s for s in composite_scenarios if s.archetype_id == arch_id]
    if arch_scenarios:
        s = arch_scenarios[0]
        print(f"  {arch_id} ({s.targets['triage_tier']}): {len(s.features)} features, "
              f"missingness={s.uncertainty_profile['missingness']:.2f}")

## 5. Stratified Scenario Instantiation

For ablation studies, we need scenarios stratified by perturbation type. The `instantiate_stratified_scenarios()` function creates separate scenario sets for each perturbation.

In [ ]:
# Generate stratified scenarios
stratified_scenarios = instantiate_stratified_scenarios(
    archetypes_df,
    n_per_archetype=8,
    seed=42,
    perturbation_config=PerturbationConfig(p_mask=0.2, noise_sigma=0.1)
)

print("Stratified Scenario Generation")
print("="*60)
print(f"{'Perturbation Type':<20} {'Scenario Count':<20} {'Avg Missingness':<20}")
print("-"*60)

for ptype, scenarios in stratified_scenarios.items():
    avg_miss = np.mean([s.uncertainty_profile.get('missingness', 0) for s in scenarios])
    print(f"{ptype:<20} {len(scenarios):<20} {avg_miss:<20.3f}")

print(f"\nTotal scenarios: {sum(len(v) for v in stratified_scenarios.values())}")
print(f"Perturbation types: {list(stratified_scenarios.keys())}")

## 6. Uncertainty Profile Visualization

Visualize how perturbations affect uncertainty across scenarios.

In [ ]:
# Plot uncertainty distribution for composite scenarios
fig, axes = plot_uncertainty_distribution(
    composite_scenarios,
    title="Uncertainty Profile Distribution (Composite Perturbation)"
)
plt.show()

print("✓ Uncertainty distribution plotted")

## 7. Perturbation Ablation Analysis

Compare the impact of different perturbation types on a custom metric.

In [ ]:
# Define metric: average feature count
def avg_feature_count(scenarios):
    return np.mean([len(s.features) for s in scenarios])

# Plot perturbation effects
baseline_only = {"baseline": stratified_scenarios["baseline"]}
perturbed_only = {k: v for k, v in stratified_scenarios.items() if k != "baseline"}

fig, ax = plot_perturbation_effects(
    stratified_scenarios["baseline"],
    perturbed_only,
    metric_fn=avg_feature_count,
    metric_name="Average Feature Count"
)
plt.show()

print("✓ Perturbation effects visualized")

## 8. Reproducibility with Seeds

Demonstrate that scenarios are deterministically reproducible given the same seed.

In [ ]:
# Generate scenarios twice with same seed
scenarios_run1 = instantiate_scenarios(
    archetypes_df,
    n_per_archetype=5,
    seed=999,
    perturbation_type="composite",
    perturbation_config=PerturbationConfig(p_mask=0.2, noise_sigma=0.1)
)

scenarios_run2 = instantiate_scenarios(
    archetypes_df,
    n_per_archetype=5,
    seed=999,  # Same seed
    perturbation_type="composite",
    perturbation_config=PerturbationConfig(p_mask=0.2, noise_sigma=0.1)
)

print("Reproducibility Check")
print("="*60)
print(f"Run 1: {len(scenarios_run1)} scenarios")
print(f"Run 2: {len(scenarios_run2)} scenarios")

# Compare first scenario from each run
s1 = scenarios_run1[0]
s2 = scenarios_run2[0]

print(f"\nScenario 0 comparison:")
print(f"  Seeds match: {s1.seed == s2.seed}")
print(f"  Features match: {s1.features == s2.features}")
print(f"  Uncertainty profiles match: {s1.uncertainty_profile == s2.uncertainty_profile}")

# Verify all scenarios match
all_match = all(
    s1.seed == s2.seed and 
    s1.features == s2.features and 
    s1.uncertainty_profile == s2.uncertainty_profile
    for s1, s2 in zip(scenarios_run1, scenarios_run2)
)

print(f"\n✓ All scenarios match: {all_match}")
print("✓ Reproducibility verified")

## 9. Summary and Key Takeaways

In this notebook, you learned:

### Core Concepts:
1. **Archetypes → Scenarios**: Controlled expansion of prototypical cases into test scenarios
2. **Perturbation Operators**: Five operators simulate different uncertainty types
3. **Uncertainty Profiles**: Quantitative measures of missingness, ambiguity, conflict, degradation
4. **Stratified Generation**: Systematic ablation studies across perturbation types
5. **Reproducibility**: Deterministic scenario generation via random seeds

### Key API Functions:

```python
# Basic instantiation
scenarios = instantiate_scenarios(
    archetypes_df,
    n_per_archetype=10,
    seed=42,
    perturbation_type="composite"
)

# Stratified instantiation for ablation
stratified = instantiate_stratified_scenarios(
    archetypes_df,
    n_per_archetype=10,
    seed=42
)
```

### Next Steps:

- **[Notebook 02](02_basics_beyond_accuracy_metrics.ipynb)**: Deep-dive into calibration metrics
- **[Notebook 03](03_coverage_risk_selective_prediction.ipynb)**: Coverage-risk analysis
- **[Notebook 04](04_harm_aware_evaluation.ipynb)**: Harm-aware evaluation
- **[Notebook 05](05_end_to_end_pipeline.ipynb)**: Complete evaluation workflow

### Production Usage:

In real evaluations, replace sample archetypes with SynDX data:

```python
from basics_cdss.scenario import load_archetypes_csv

archetypes = load_archetypes_csv("data/syndx_archetypes.csv")
scenarios = instantiate_scenarios(archetypes, n_per_archetype=50, seed=42)
```